In [ ]:
from aind_behavior_vr_foraging.data_contract import dataset
from aind_behavior_vr_foraging_nwb.processing import TrialTableProcessor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

all_sessions: list[pd.DataFrame] = []
for session in Path(r"./data").iterdir():
    this_ds = dataset(session)
    processor = TrialTableProcessor(this_ds)
    data = processor.process_to_sites()
    df = pd.json_normalize([m.model_dump() for m in data])
    acquisition_time = this_ds["Behavior"]["InputSchemas"]["Session"].data.date
    df["session_id"] = session.name
    df["date"] = acquisition_time
    df["subject"] = this_ds["Behavior"]["InputSchemas"]["Session"].data.subject
    all_sessions.append(df)
df_raw = pd.concat(all_sessions, ignore_index=True)

In [ ]:
trimmed_sessions = []
for session_id, session_df in df_raw.groupby("session_id"):
    trimmed = session_df.iloc[: len(session_df) - len(session_df) // 3].copy()
    trimmed["session_id"] = session_id
    trimmed_sessions.append(trimmed)

df = pd.concat(trimmed_sessions, ignore_index=True)
df = df[df["site_label"] == "RewardSite"]

In [ ]:
transition_matrix = this_ds["Behavior"]["InputSchemas"]["TaskLogic"].data
transition_matrix = (
    np.array(
        transition_matrix.task_parameters.environment.blocks[
            0
        ].environment_statistics.transition_matrix
    )
    > 0
)

fig = plt.Figure(figsize=(8, 6))
plt.imshow(transition_matrix, cmap="viridis")
plt.title("Transition Matrix")
plt.xlabel("From State")
plt.ylabel("To State")
plt.show()

In [ ]:
# Presentation plot: reward-site stop probability across sessions
# Set date range here (inclusive). Use None for open-ended bounds.
START_DATE = None  # "2026-01-31"
END_DATE = None # "2026-02-10"
MIN_FONT_SIZE = 20

required_columns = {"session_id", "date", "patch_label", "has_choice"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(
        f"Cell 2 must be re-run before plotting. Missing columns: {sorted(missing_columns)}"
    )

plot_df = df.loc[:, ["session_id", "date", "patch_label", "has_choice"]].copy()
plot_df["date"] = (
    pd.to_datetime(plot_df["date"], utc=True).dt.tz_convert(None).dt.normalize()
)

start_ts = (
    pd.to_datetime(START_DATE).normalize()
    if START_DATE is not None
    else plot_df["date"].min()
)
end_ts = (
    pd.to_datetime(END_DATE).normalize()
    if END_DATE is not None
    else plot_df["date"].max()
)

plot_df = plot_df.loc[
    (plot_df["date"] >= start_ts) & (plot_df["date"] <= end_ts)
].copy()
if plot_df.empty:
    raise ValueError("No sessions found in the selected date range.")

session_lookup = (
    plot_df[["session_id", "date"]]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)
session_lookup["session_number"] = range(len(session_lookup))

plot_df = plot_df.merge(
    session_lookup[["session_id", "session_number"]], on="session_id", how="left"
)

p_stop_by_session = (
    plot_df.groupby(["session_number", "patch_label"], as_index=False)["has_choice"]
    .mean()
    .rename(columns={"has_choice": "p_stop"})
)

style_map = {
    "A": {"color": "#66c2a5", "linestyle": "-", "marker": "o"},
    "B1": {"color": "#fc8d62", "linestyle": "-", "marker": "o"},
    "B2": {"color": "#fc8d62", "linestyle": "--", "marker": "s"},
    "C1": {"color": "#8da0cb", "linestyle": "-", "marker": "o"},
    "C2": {"color": "#8da0cb", "linestyle": "--", "marker": "s"},
}
plot_order = [
    label
    for label in ["A", "B1", "B2", "C1", "C2"]
    if label in p_stop_by_session["patch_label"].unique()
]

fig, ax = plt.subplots(figsize=(8, 4))

for patch_label in plot_order:
    group_df = p_stop_by_session.loc[
        p_stop_by_session["patch_label"] == patch_label
    ].sort_values("session_number")
    style = style_map[patch_label]
    ax.plot(
        group_df["session_number"],
        group_df["p_stop"],
        label=patch_label,
        color=style["color"],
        linestyle=style["linestyle"],
        marker=style["marker"],
        linewidth=3,
        markersize=9,
        markeredgecolor="white",
        markeredgewidth=1.0,
    )

ax.set_xlabel("Session Number", fontsize=MIN_FONT_SIZE)
ax.set_ylabel("P(Stop)", fontsize=MIN_FONT_SIZE)
ax.set_ylim(-0.1, 1.1)
ax.set_xticks(session_lookup["session_number"])
ax.set_yticks([0, 0.5, 1])
ax.tick_params(axis="both", labelsize=MIN_FONT_SIZE)
ax.grid(axis="y", alpha=0.25, linewidth=0.8)
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(
    title_fontsize=15,
    fontsize=15,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    borderaxespad=0.0,
)

fig.tight_layout(rect=[0, 0, 0.8, 1])
plt.savefig("./local/reward_site_stop_probability_across_sessions.svg")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch


def plot_state_dag(
    session_df, transition_matrix, node_size=1400, figsize=(13, 6), title=None
):
    session_df = session_df.copy()
    if "site_label" in session_df.columns:
        session_df = session_df[session_df["site_label"] == "RewardSite"]

    p_stop = session_df.groupby("patch_label")["has_choice"].mean()

    # Concatenated two-stage layout with terminal A1 on the right:
    # A1 -> B1/B2 -> C1/C2 -> A2 -> C3/C4 -> B3/B4 -> A1
    node_pos = {
        "A1_start": (0.10, 0.50),
        "B1": (0.30, 0.50),
        "B2": (0.45, 0.70),
        "C2": (0.70, 0.30),
        "C1": (0.70, 0.70),
        "A2": (0.90, 0.50),
        "C3": (1.10, 0.50),
        "C4": (1.25, 0.70),
        "B4": (1.50, 0.30),
        "B3": (1.50, 0.70),
        "A1_end": (1.72, 0.50),
    }

    letter_colors = {
        "A": "#66c2a5",
        "B": "#fc8d62",
        "C": "#8da0cb",
    }

    base_labels = ["A", "B1", "B2", "C2", "C1"]
    label_to_idx = {label: i for i, label in enumerate(base_labels)}

    # Some sessions may not have all labels in this window.
    p_stop = p_stop.reindex(base_labels).fillna(0.0)

    transition_matrix = np.asarray(transition_matrix)
    if transition_matrix.shape[0] < len(base_labels) or transition_matrix.shape[1] < len(
        base_labels
    ):
        raise ValueError(
            "transition_matrix does not match the expected patch label layout"
        )

    # Reuse base task probabilities/structure for renamed nodes.
    node_to_base = {
        "A1_start": "A",
        "B1": "B1",
        "B2": "B2",
        "C2": "C2",
        "C1": "C1",
        "A2": "A",
        "C3": "B1",
        "C4": "B2",
        "B4": "C2",
        "B3": "C1",
        "A1_end": "A",
    }

    def node_probability(label):
        return float(np.clip(p_stop[node_to_base[label]], 0.0, 1.0))

    def blended_fill(base_color, probability):
        base_rgb = np.array(to_rgb(base_color))
        white_rgb = np.ones(3)
        return tuple((1.0 - probability) * white_rgb + probability * base_rgb)

    def draw_straight_arrow(ax, start, end):
        x0, y0 = start
        x1, y1 = end
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle="-|>",
                color="#2f2f2f",
                lw=1.2,
                mutation_scale=10,
                shrinkA=np.sqrt(node_size) * 0.02,
                shrinkB=np.sqrt(node_size) * 0.02,
                connectionstyle="arc3,rad=0.0",
            ),
            zorder=1,
        )

    def draw_single_kink_arrow(ax, start, end, kink_x, kink_y):
        x0, y0 = start
        x1, y1 = end
        ax.plot(
            [x0, kink_x, x1], [y0, kink_y, kink_y], color="#2f2f2f", lw=1.2, zorder=1
        )
        arrow = FancyArrowPatch(
            (kink_x, kink_y),
            (x1, y1),
            arrowstyle="-|>",
            color="#2f2f2f",
            lw=1.2,
            mutation_scale=10,
            zorder=1,
        )
        ax.add_patch(arrow)

    fig, ax = plt.subplots(figsize=figsize)

    stages = [
        (
            ["A1_start", "B1", "B2", "C2", "C1"],
            "A1_start",
            ("B1", "C2", "B2", "C2"),
        ),
        (["A2", "C3", "C4", "B4", "B3"], "A2", ("C3", "B4", "C4", "B4")),
    ]

    for stage_nodes, stage_entry, kink_rule in stages:
        kink_src, kink_dst, kink_x_ref, kink_y_ref = kink_rule

        for src in stage_nodes:
            i = label_to_idx[node_to_base[src]]

            for dst in stage_nodes:
                if dst == stage_entry:
                    continue

                j = label_to_idx[node_to_base[dst]]
                if transition_matrix[i, j] <= 0:
                    continue

                start = node_pos[src]
                end = node_pos[dst]

                if src == kink_src and dst == kink_dst:
                    kink_x = node_pos[kink_x_ref][0]
                    kink_y = node_pos[kink_y_ref][1]
                    draw_single_kink_arrow(ax, start, end, kink_x=kink_x, kink_y=kink_y)
                else:
                    draw_straight_arrow(ax, start, end)

    # Bridge between stages: C1/C2 -> A2
    for src_label in ["C2", "C1"]:
        draw_straight_arrow(ax, node_pos[src_label], node_pos["A2"])

    # New terminal bridge: B3/B4 -> A1 (terminal, no loop-back)
    for src_label in ["B4", "B3"]:
        draw_straight_arrow(ax, node_pos[src_label], node_pos["A1_end"])

    for label, (x, y) in node_pos.items():
        display_label = "A1" if label in {"A1_start", "A1_end"} else label
        base_color = letter_colors.get(display_label[0], "gray")
        probability = node_probability(label)
        face_color = blended_fill(base_color, probability)

        ax.scatter(
            x,
            y,
            s=node_size,
            facecolors=[face_color],
            edgecolors=base_color,
            linewidths=2,
            zorder=3,
        )
        ax.text(
            x,
            y,
            display_label,
            ha="center",
            va="center",
            fontsize=10,
            color="#222222",
            zorder=4,
        )

    state_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markersize=12,
            markerfacecolor=blended_fill(letter_colors[state], 1.0),
            markeredgecolor=letter_colors[state],
            markeredgewidth=2,
            label=f"State {state}",
        )
        for state in ["A", "B", "C"]
    ]
    pstop_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markersize=12,
            markerfacecolor=blended_fill(letter_colors["A"], probability),
            markeredgecolor=letter_colors["A"],
            markeredgewidth=2,
            label=f"{int(probability * 100)}%",
        )
        for probability in [0.0, 0.5, 1.0]
    ]

    state_legend = ax.legend(
        handles=state_handles,
        title="State",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 0.92),
    )
    ax.add_artist(state_legend)
    ax.legend(
        handles=pstop_handles,
        title="P(Stop)",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 0.52),
    )

    if title:
        ax.set_title(title, fontsize=11)

    ax.set_xlim(-0.05, 1.90)
    ax.set_ylim(-0.05, 1.05)
    ax.axis("off")
    plt.tight_layout()
    if title is None:
        out_name = "session_dag"
    else:
        out_name = f"session_{title.replace(' ', '_')}"
    plt.savefig(f"./local/{out_name}.svg")
    plt.close(fig)


# Generate one DAG for each session shown in the previous plot cell.
if "session_lookup" not in globals():
    raise ValueError("Run the previous cell first to create session_lookup.")

for _, row in session_lookup.sort_values("session_number").iterrows():
    session_number = int(row["session_number"])
    session_id = row["session_id"]
    this_session_df = df[df["session_id"] == session_id]

    pstop_summary = (
        this_session_df.groupby("patch_label")["has_choice"]
        .mean()
        .reindex(["A", "B1", "B2", "C1", "C2"])
        .round(3)
    )
    print(f"Session {session_number} | {session_id}")
    print(pstop_summary.to_dict())

    if "C4" in this_session_df["patch_label"].unique():
        print("Contains C4 patch label.")
        plot_state_dag(
            this_session_df,
            transition_matrix,
            title=f"Session {session_number}",
        )

In [ ]:
df["patch_label"].unique()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from matplotlib.path import Path as MPath


def plot_state_dag(
    session_df, transition_matrix, node_size=1500, figsize=(8, 6), title=None
):
    session_df = session_df.copy()
    if "site_label" in session_df.columns:
        session_df = session_df[session_df["site_label"] == "RewardSite"]

    p_stop = session_df.groupby("patch_label")["has_choice"].mean()

    node_pos = {
        "A": (0.10, 0.50),
        "B1": (0.30, 0.50),
        "B2": (0.45, 0.70),
        "C2": (0.70, 0.30),
        "C1": (0.70, 0.70),
        "A_end": (0.90, 0.50),
    }

    letter_colors = {
        "A": "#66c2a5",
        "B": "#fc8d62",
        "C": "#8da0cb",
    }
    patch_labels = ["A", "B1", "B2", "C2", "C1"]
    label_to_idx = {label: i for i, label in enumerate(patch_labels)}

    missing_labels = [label for label in patch_labels if label not in p_stop.index]
    if missing_labels:
        raise ValueError(f"Missing patch labels in dataframe: {missing_labels}")

    transition_matrix = np.asarray(transition_matrix)
    if transition_matrix.shape[0] < len(patch_labels) or transition_matrix.shape[
        1
    ] < len(patch_labels):
        raise ValueError(
            "transition_matrix does not match the expected patch label layout"
        )

    def node_probability(label):
        return float(
            np.clip(p_stop["A"] if label == "A_end" else p_stop[label], 0.0, 1.0)
        )

    def blended_fill(base_color, probability):
        base_rgb = np.array(to_rgb(base_color))
        white_rgb = np.ones(3)
        return tuple((1.0 - probability) * white_rgb + probability * base_rgb)

    fig, ax = plt.subplots(figsize=figsize)

    for src, (x0, y0) in node_pos.items():
        if src == "A_end" or src not in label_to_idx:
            continue
        i = label_to_idx[src]

        for dst, (x1, y1) in node_pos.items():
            if dst in {"A", "A_end"} or dst not in label_to_idx:
                continue

            j = label_to_idx[dst]
            if transition_matrix[i, j] > 0:
                # Special handling for B1 to C2: add a kink
                if src == "B1" and dst == "C2":
                    # Create a kinked path: B1 -> (B2_x, C2_y) -> C2
                    kink_x = node_pos["B2"][0]
                    kink_y = node_pos["C2"][1]  # Use C2's y position for the kink
                    
                    # Draw the kinked line with single kink at (kink_x, kink_y)
                    ax.plot([x0, kink_x, x1], [y0, kink_y, kink_y], color="#2f2f2f", lw=1.2, zorder=1)
                    
                    # Add arrowhead at the end
                    arrow = FancyArrowPatch(
                        (kink_x, kink_y), (x1, y1),
                        arrowstyle="-|>",
                        color="#2f2f2f",
                        lw=1.2,
                        mutation_scale=10,
                        zorder=1,
                    )
                    ax.add_patch(arrow)
                else:
                    ax.annotate(
                        "",
                        xy=(x1, y1),
                        xytext=(x0, y0),
                        arrowprops=dict(
                            arrowstyle="-|>",
                            color="#2f2f2f",
                            lw=1.2,
                            mutation_scale=10,
                            shrinkA=np.sqrt(node_size) * 0.02,
                            shrinkB=np.sqrt(node_size) * 0.02,
                            connectionstyle="arc3,rad=0.0",
                        ),
                        zorder=1,
                    )

    for src_label in ["C2", "C1"]:
        x0, y0 = node_pos[src_label]
        x1, y1 = node_pos["A_end"]
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle="-|>",
                color="#2f2f2f",
                lw=1.2,
                mutation_scale=10,
                shrinkA=np.sqrt(node_size) * 0.02,
                shrinkB=np.sqrt(node_size) * 0.02,
                connectionstyle="arc3,rad=0.0",
            ),
            zorder=1,
        )

    for label, (x, y) in node_pos.items():
        display_label = "A" if label == "A_end" else label
        base_color = letter_colors.get(display_label[0], "gray")
        probability = node_probability(label)
        face_color = blended_fill(base_color, probability)

        ax.scatter(
            x,
            y,
            s=node_size,
            facecolors=[face_color],
            edgecolors=base_color,
            linewidths=2,
            zorder=3,
        )
        ax.text(
            x,
            y,
            display_label,
            ha="center",
            va="center",
            fontsize=10,
            color="#222222",
            zorder=4,
        )

    state_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markersize=12,
            markerfacecolor=blended_fill(letter_colors[state], 1.0),
            markeredgecolor=letter_colors[state],
            markeredgewidth=2,
            label=f"State {state}",
        )
        for state in ["A", "B", "C"]
    ]
    pstop_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markersize=12,
            markerfacecolor=blended_fill(letter_colors["A"], probability),
            markeredgecolor=letter_colors["A"],
            markeredgewidth=2,
            label=f"{int(probability * 100)}%",
        )
        for probability in [0.0, 0.5, 1.0]
    ]

    state_legend = ax.legend(
        handles=state_handles,
        title="State",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 0.92),
    )
    ax.add_artist(state_legend)
    ax.legend(
        handles=pstop_handles,
        title="P(Stop)",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 0.52),
    )

    if title:
        ax.set_title(title, fontsize=11)

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"./local/session_{title.replace(' ', '_')}.svg")


# Generate one DAG for each session shown in the previous plot cell.
if "session_lookup" not in globals():
    raise ValueError("Run the previous cell first to create session_lookup.")

for _, row in session_lookup.sort_values("session_number").iterrows():
    session_number = int(row["session_number"])
    session_id = row["session_id"]
    this_session_df = df[df["session_id"] == session_id]

    pstop_summary = (
        this_session_df.groupby("patch_label")["has_choice"]
        .mean()
        .reindex(["A", "B1", "B2", "C1", "C2"])
        .round(3)
    )
    print(f"Session {session_number} | {session_id}")
    print(pstop_summary.to_dict())

    plot_state_dag(
        this_session_df,
        transition_matrix,
        title=f"Session {session_number}",
    )
